# Northwind Data Visualization - Interactive Dashboard

This notebook contains interactive visualizations using Plotly, including delivery statistics and 3D analysis.

In [12]:
# Install required packages if not already installed
!pip install plotly nbformat pandas --quiet

In [13]:

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import os

# Ensure figures dir exists
os.makedirs("../figures", exist_ok=True)

# --- Global Design Settings ---
# Set default template
pio.templates.default = "plotly_white"

# Define Custom Color Palette (Modern Tech Theme)
COLORS = {
    'primary': '#6c5ce7',    # Deep Purple
    'secondary': '#00cec9',  # Robinson Teal
    'accent': '#fdcb6e',     # Warm Yellow
    'delivered': '#6c5ce7',  # Purple for Delivered
    'not_delivered': '#00cec9', # Teal for Not Delivered
    'background': '#ffffff', 
    'text': '#2d3436'
}


In [14]:
# Load the extracted data from CSVs
extracted_dir = "../data/extracted"

try:
    fact_orders = pd.read_csv(os.path.join(extracted_dir, "FactOrders.csv"))
    dim_customers = pd.read_csv(os.path.join(extracted_dir, "DimCustomer.csv"))
    dim_employees = pd.read_csv(os.path.join(extracted_dir, "DimEmployee.csv"))
    dim_date = pd.read_csv(os.path.join(extracted_dir, "DimDate.csv"))

    # Merge DataFrames to recreate the analysis dataset
    # 1. Join Orders with Customers
    df = fact_orders.merge(dim_customers, on="CustomerId", how="left")
    
    # 2. Join with Employees (will create _x (Customer) and _y (Employee) suffixes)
    df = df.merge(dim_employees, on="EmployeeId", how="left")
    
    # 3. Join with Date
    df = df.merge(dim_date, on="DateId", how="left")

    df['FullDate'] = pd.to_datetime(df['FullDate'])
    print("Data loaded and merged successfully from new CSVs.")
    print(f"Total records: {len(df)}")
    print(df.head())

except FileNotFoundError as e:
    print(f"Error loading data: {e}. Please run the extraction scripts first.")

Data loaded and merged successfully from new CSVs.
Total records: 48
   OrderId  CustomerId  EmployeeId    DateId  DeliveredFlag CompanyName  \
0       30          27           9  20060115              1  Company AA   
1       31           4           3  20060120              1   Company D   
2       32          12           4  20060122              1   Company L   
3       33           8           6  20060130              1   Company H   
4       34           4           9  20060206              1   Company D   

      City_x Country_x FirstName        LastName    City_y Country_y  \
0  Las Vegas       USA      Anne  Hellung-Larsen   Seattle       USA   
1   New York       USA       Jan           Kotas   Redmond       USA   
2  Las Vegas       USA    Mariya       Sergienko  Kirkland       USA   
3   Portland       USA   Michael         Neipper   Redmond       USA   
4   New York       USA      Anne  Hellung-Larsen   Seattle       USA   

    FullDate  Day  Month MonthName  
0 2006-01-

## 1. Delivery Status Overview

In [15]:
if 'df' in locals():
    delivery_counts = df['DeliveredFlag'].value_counts()
    
    # Create Donut Chart with Pulled Slices
    fig = go.Figure(data=[go.Pie(
        labels=['Delivered', 'Not Delivered'],
        values=[delivery_counts.get(1, 0), delivery_counts.get(0, 0)],
        hole=0.4, 
        pull=[0, 0.1], # Pull the 'Not Delivered' slice slightly
        marker=dict(colors=[COLORS['delivered'], COLORS['not_delivered']]), 
        textinfo='label+percent',
        rotation=90
    )])
    
    fig.update_layout(
        title=dict(text='Order Delivery Status', x=0.5, xanchor='center'),
        font=dict(size=14, family="Segoe UI, sans-serif"),
        height=500,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    try:
        fig.write_html("../figures/delivery_stats_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")


## 2. Orders by Country (Interactive)

In [16]:
if 'df' in locals():
    # Group by Country and Delivery Status
    country_delivery = df.groupby(['Country_x', 'DeliveredFlag']).size().reset_index(name='Count')
    # Calculate percentage within each country
    country_totals = country_delivery.groupby('Country_x')['Count'].transform('sum')
    country_delivery['Percentage'] = (country_delivery['Count'] / country_totals) * 100
    country_delivery['Status'] = country_delivery['DeliveredFlag'].map({1: 'Delivered', 0: 'Not Delivered'})
    
    fig = px.bar(
        country_delivery,
        x='Country_x',
        y='Percentage',
        title='Total Orders by Country (100% Stacked)',
        color='Status',
        color_discrete_map={'Delivered': COLORS['delivered'], 'Not Delivered': COLORS['not_delivered']},
        barmode='stack',
        labels={'Country_x': 'Country', 'Percentage': 'Percentage of Orders (%)'}
    )
    
    fig.update_layout(
        xaxis_title='Country',
        yaxis_title='Percentage of Orders (%)',
        yaxis=dict(ticksuffix='%'),
        height=600,
        title=dict(x=0.5, xanchor='center'),
        font=dict(family="Segoe UI, sans-serif")
    )
    
    try:
        fig.write_html("../figures/orders_by_country_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")


## 3. Orders by Employee (Interactive)

In [17]:
if 'df' in locals():
    df['EmployeeName'] = df['FirstName'] + ' ' + df['LastName']
    # Group by Employee and Delivery Status
    employee_delivery = df.groupby(['EmployeeName', 'DeliveredFlag']).size().reset_index(name='Count')
    # Calculate percentage
    emp_totals = employee_delivery.groupby('EmployeeName')['Count'].transform('sum')
    employee_delivery['Percentage'] = (employee_delivery['Count'] / emp_totals) * 100
    employee_delivery['Status'] = employee_delivery['DeliveredFlag'].map({1: 'Delivered', 0: 'Not Delivered'})
    
    fig = px.bar(
        employee_delivery,
        y='EmployeeName',
        x='Percentage',
        orientation='h',
        title='Orders by Employee (100% Stacked)',
        color='Status',
        color_discrete_map={'Delivered': COLORS['delivered'], 'Not Delivered': COLORS['not_delivered']},
        barmode='stack',
        labels={'EmployeeName': 'Employee', 'Percentage': 'Percentage of Orders (%)'}
    )
    
    fig.update_layout(
        xaxis_title='Percentage of Orders (%)',
        yaxis_title='Employee',
        xaxis=dict(ticksuffix='%'),
        height=600,
        title=dict(x=0.5, xanchor='center'),
        font=dict(family="Segoe UI, sans-serif")
    )
    
    try:
        fig.write_html("../figures/orders_by_employee_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")


## 4. Monthly Orders Trend (Interactive)

In [18]:
if 'df' in locals():
    # Create YearMonth column for grouping
    df['YearMonth'] = df['FullDate'].dt.to_period('M').astype(str)
    
    # 4. Monthly Order Trend
    monthly_trend = df.groupby('YearMonth').size().reset_index(name='Count')
    monthly_trend = monthly_trend.sort_values('YearMonth')
    
    # Area Chart for Trend
    fig = px.area(monthly_trend, x='YearMonth', y='Count', 
                  title='Monthly Order Trend',
                  markers=True,
                  labels={'YearMonth': 'Month', 'Count': 'Number of Orders'})
    
    fig.update_traces(line_color=COLORS['primary'], fill='tozeroy')
    
    fig.update_layout(
        font=dict(family="Segoe UI, sans-serif"),
        title=dict(x=0.5, xanchor='center'),
        height=500,
        plot_bgcolor='rgba(0,0,0,0)',
        xaxis=dict(showgrid=False),
        yaxis=dict(showgrid=True, gridcolor='LightGray')
    )
    
    try:
        fig.write_html("../figures/monthly_trend_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")


## 5. 3D Interactive Visualization: Orders by Month and Country

In [19]:
# --- MOCK DATA GENERATION (1996-2005) ---
# The source data only contains 2006. We generate mock history for visualization purposes.
import numpy as np

if 'df' in locals():
    print("Generating mock historical data (1996-2005)...")
    
    # Get existing unique entities to maintain referential consistency
    customers = df['CompanyName'].unique()
    employees = df['LastName'].unique()
    
    mock_rows = []
    # Generate ~500 mock orders scattered across the years
    for _ in range(500):
        # Random Year between 1996 and 2005
        year = np.random.randint(1996, 2006)
        # Random Month and Day
        month = np.random.randint(1, 13)
        day = np.random.randint(1, 28)
        mock_date = pd.Timestamp(year=year, month=month, day=day)
        
        row = {
            'CompanyName': np.random.choice(customers),
            'LastName': np.random.choice(employees),
            'FullDate': mock_date,
            'Year': year,
            'OrderCount': 1, # Placeholder
            'DeliveredFlag': np.random.choice([0, 1]), # Add this for delivery stats
            'Country_x': 'USA' # Default mock country to avoid NaN in country stats
        }
        mock_rows.append(row)
    
    mock_df = pd.DataFrame(mock_rows)
    
    # Ensure original df has 'Year' computed if not already
    if 'Year' not in df.columns:
        df['FullDate'] = pd.to_datetime(df['FullDate'])
        df['Year'] = df['FullDate'].dt.year
    
    # FIXED LINE: Do not subset df. Concatenate everything. 
    # Mock data will have NaNs for columns we didn't mock, which is fine.
    df = pd.concat([df, mock_df], ignore_index=True)
    
    print(f"Added {len(mock_df)} mock records. Total records: {len(df)}")


Generating mock historical data (1996-2005)...
Added 500 mock records. Total records: 548


In [20]:
# OLAP 3D Visualization: Orders by Customer, Employee, Date (with Year Selection)
if 'df' in locals():
    # Ensure FullDate is datetime
    df['FullDate'] = pd.to_datetime(df['FullDate'])
    
    # Extract Year for animation/selection
    df['Year'] = df['FullDate'].dt.year
    
    # Aggregate data: Group by Year, Customer, Employee, Date
    olap_df = df.groupby(['Year', 'CompanyName', 'LastName', 'FullDate']).size().reset_index(name='OrderCount')
    olap_df = olap_df.sort_values('Year')
    
    # Plotly Express 3D Scatter with Animation Frame
    fig_3d = px.scatter_3d(
        olap_df,
        x='CompanyName',
        y='LastName',
        z='FullDate',
        size='OrderCount',
        color='OrderCount',
        color_continuous_scale='Plasma', # Keep Plasma
        animation_frame='Year',
        animation_group='CompanyName',
        title='OLAP View: Orders by Customer, Employee, Date (Yearly Selection)',
        labels={'CompanyName': 'Customer', 'LastName': 'Employee', 'FullDate': 'Date'}
    )
    
    fig_3d.update_layout(
        scene=dict(
            xaxis_title='Customer',
            yaxis_title='Employee',
            zaxis_title='Date',
            xaxis=dict(backgroundcolor="rgba(0,0,0,0)", gridcolor='lightgray'),
            yaxis=dict(backgroundcolor="rgba(0,0,0,0)", gridcolor='lightgray'),
            zaxis=dict(backgroundcolor="rgba(0,0,0,0)", gridcolor='lightgray'),
        ),
        font=dict(family="Segoe UI, sans-serif"),
        title=dict(x=0.5, xanchor='center'),
        height=800,
        margin=dict(r=0, l=0, b=0, t=50)
    )
    
    try:
        fig_3d.write_html("../figures/3d_orders_notebook.html")
        fig_3d.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")


## 7. Comprehensive Dashboard

In [21]:
if 'df' in locals():
    delivery_counts = df['DeliveredFlag'].value_counts()
    
    # Create Donut Chart with Pulled Slices
    fig = go.Figure(data=[go.Pie(
        labels=['Delivered', 'Not Delivered'],
        values=[delivery_counts.get(1, 0), delivery_counts.get(0, 0)],
        hole=0.4, 
        pull=[0, 0.1], # Pull the 'Not Delivered' slice slightly
        marker=dict(colors=[COLORS['delivered'], COLORS['not_delivered']]), 
        textinfo='label+percent',
        rotation=90
    )])
    
    fig.update_layout(
        title=dict(text='Order Delivery Status', x=0.5, xanchor='center'),
        font=dict(size=14, family="Segoe UI, sans-serif"),
        height=500,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    try:
        fig.write_html("../figures/delivery_stats_notebook.html")
        fig.show()
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")


## 4. Detailed Delivery Analysis
Analysis of delivery efficiency by Employee and Country.

In [22]:
# Delivery Status Analysis (100% Stacked)
if 'df' in locals():
    # 1. Delivery Status by Employee (Horizontal 100%)
    emp_delivery = df.groupby(['LastName', 'DeliveredFlag']).size().reset_index(name='Count')
    # Calculate percentage
    emp_totals = emp_delivery.groupby('LastName')['Count'].transform('sum')
    emp_delivery['Percentage'] = (emp_delivery['Count'] / emp_totals) * 100
    emp_delivery['Status'] = emp_delivery['DeliveredFlag'].map({1: 'Delivered', 0: 'Not Delivered'})
    
    fig_emp = px.bar(emp_delivery, y='LastName', x='Percentage', color='Status', 
                     title='Delivery Status by Employee (100% Stacked)', 
                     orientation='h', # Horizontal
                     labels={'LastName': 'Employee', 'Percentage': 'Percentage of Orders (%)'},
                     barmode='stack',
                     color_discrete_map={'Delivered': COLORS['delivered'], 'Not Delivered': COLORS['not_delivered']})
    
    # 2. Delivery Status by Country (Horizontal 100%)
    country_delivery = df.groupby(['Country_x', 'DeliveredFlag']).size().reset_index(name='Count')
    # Calculate percentage
    country_totals = country_delivery.groupby('Country_x')['Count'].transform('sum')
    country_delivery['Percentage'] = (country_delivery['Count'] / country_totals) * 100
    country_delivery['Status'] = country_delivery['DeliveredFlag'].map({1: 'Delivered', 0: 'Not Delivered'})
    
    fig_country = px.bar(country_delivery, y='Country_x', x='Percentage', color='Status', 
                         title='Delivery Status by Country (100% Stacked)', 
                         orientation='h', # Horizontal
                         labels={'Country_x': 'Country', 'Percentage': 'Percentage of Orders (%)'},
                         barmode='stack',
                         color_discrete_map={'Delivered': COLORS['delivered'], 'Not Delivered': COLORS['not_delivered']})
    
    # Display figures
    for fig in [fig_emp, fig_country]:
        fig.update_layout(
            height=500,
            font=dict(family="Segoe UI, sans-serif"),
            title=dict(x=0.5, xanchor='center'),
            plot_bgcolor='rgba(0,0,0,0)',
            yaxis={'categoryorder':'total ascending'}, # Sort bars
            xaxis=dict(ticksuffix='%') # Add % sign
        )
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
        fig.update_yaxes(showgrid=False)
    
    try:
        fig_emp.show()
        fig_country.show()
        fig_emp.write_html("../figures/delivery_by_employee.html")
        fig_country.write_html("../figures/delivery_by_country.html")
    except Exception as e:
        print(f"Error displaying/saving plot: {e}")
